# 11. CU-BEMS: hourly series (task T31)
Reads the 14 one-minute CU-BEMS files in `data/cu_bems/`, keeps the power columns (kW), converts to hourly energy (kWh) and writes `data/cu_bems_hourly/cu_bems_hourly_wide.csv.gz`.

**Rules.** An hour counts for a device only if it has at least 45 of 60 minutes. A floor-hour is valid only if every device on the floor passes. Invalid hours stay missing (no filling in). `total_6floors` is the sum of floors 1-5 and 7, because the 2019 file for floor 6 stops on 22 Oct 2019.

Run from the `notebooks/` folder.

In [ ]:
import pandas as pd, numpy as np, glob, os
from pathlib import Path
SRC=Path('../data/cu_bems'); OUT=Path('../data/cu_bems_hourly'); OUT.mkdir(exist_ok=True)
MIN_MINUTES=45   # an hour counts for a device only if it has at least 45 of 60 minutes
rows=[];stats=[]
for f in sorted(SRC.glob('20*Floor*.csv')):
    b=f.name; fl=int(b.split('Floor')[1][0]); yr=int(b[:4])
    cols=[c for c in pd.read_csv(f,nrows=0).columns if c.endswith('(kW)')]   # power columns only, not temperature/humidity/light sensors
    d=pd.read_csv(f,usecols=['Date']+cols,dtype={**{c:'float32' for c in cols},'Date':str})
    d['Date']=pd.to_datetime(d['Date'],errors='coerce'); d=d.dropna(subset=['Date'])   # drops a stray bad row in 2019Floor6.csv
    d=d.drop_duplicates('Date').set_index('Date')
    g=d.groupby(d.index.floor('h')); m=g.mean(); n=g.count(); ok=n>=MIN_MINUTES; m=m.where(ok)
    for c in cols: stats.append((fl,yr,c,len(d),float(d[c].isna().mean()),float((d[c]==0).mean())))
    o=pd.DataFrame({'floor':fl,'kwh_sum_available':m.sum(axis=1,min_count=1),'n_dev':len(cols),'frac_dev_ok':ok.mean(axis=1),'all_dev_ok':ok.all(axis=1)})
    o.index.name='hour'; rows.append(o.reset_index()); print(b,len(d),d.index.min(),d.index.max(),len(cols))
r=pd.concat(rows)
pd.DataFrame(stats,columns=['floor','year','device','minutes','frac_missing','frac_zero']).to_csv(OUT/'device_stats.csv',index=False)
# floor hourly energy (kWh) = sum of device mean kW over the hour; an hour is valid only if every device had >=45 minutes
r['kwh']=r.kwh_sum_available.where(r.all_dev_ok)
grid=pd.date_range('2018-07-01','2019-12-31 23:00',freq='h')
w=r.pivot(index='hour',columns='floor',values='kwh').reindex(grid); w.columns=[f'floor{c}' for c in w.columns]
w['total_6floors']=w[['floor1','floor2','floor3','floor4','floor5','floor7']].sum(axis=1,min_count=6)   # floor 6 file ends 22 Oct 2019
w.to_csv(OUT/'cu_bems_hourly_wide.csv.gz')
print(w.notna().mean().round(3))
